In [1]:
# ── SPATIAL QUERY 1 — Markets per district ────────────────────────────────────
import geopandas as gpd
import pandas as pd

districts = gpd.read_file('malawi_districts_risk.geojson')
markets   = gpd.read_file('malawi_markets.geojson')
spikes    = gpd.read_file('malawi_spikes.geojson')

markets_in_districts = gpd.sjoin(
    markets,
    districts[['NAME_1', 'risk_score', 'critical_count', 'geometry']],
    how='left',
    predicate='within'
)

mkt_count = (
    markets_in_districts
    .groupby('NAME_1')
    .agg(
        market_count   = ('market', 'count'),
        avg_spike_rate = ('spike_rate_pct', 'mean'),
        risk_score     = ('risk_score', 'first')
    )
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== QUERY 1: MARKETS PER DISTRICT ===')
print(mkt_count.to_string(index=False))

=== QUERY 1: MARKETS PER DISTRICT ===
    NAME_1  market_count  avg_spike_rate  risk_score
  Machinga             8        9.877500         436
   Mulanje             5       10.904000         277
     Zomba             5        8.646000         262
    Nsanje             6        8.935000         252
  Chikwawa             5       11.226000         217
      Dowa             7        6.882857         199
    Ntcheu             5        7.960000         185
  Blantyre             4        6.597500         181
   Ntchisi             5        8.450000         175
  Phalombe             5        8.364000         167
    Thyolo             5       10.712000         156
    Mzimba             7        5.418571         126
    Balaka             5        7.954000         126
  Mangochi             7        5.875714         119
   Mchinji             3        8.056667         108
   Karonga             2       10.805000         104
    Rumphi             2       11.185000          92
   Kasun

In [2]:
# ── SPATIAL QUERY 2 — Markets within 100km of Blantyre ───────────────────────
districts_utm = districts.to_crs('EPSG:32736')
markets_utm   = markets.to_crs('EPSG:32736')

blantyre          = districts_utm[districts_utm['NAME_1'] == 'Blantyre'].copy()
blantyre_centroid = blantyre.geometry.centroid.iloc[0]
buffer_100km      = blantyre_centroid.buffer(100_000)

markets_utm['in_100km']    = markets_utm.geometry.within(buffer_100km)
nearby                     = markets_utm[markets_utm['in_100km']].copy()
nearby['distance_km']      = (
    nearby.geometry.distance(blantyre_centroid) / 1000
).round(1)

print('=== QUERY 2: MARKETS WITHIN 100KM OF BLANTYRE ===')
print(f'Total markets     : {len(markets_utm)}')
print(f'Within 100km      : {len(nearby)}')
print()
print(nearby[['market','district','distance_km','total_spikes','spike_rate_pct']]
      .sort_values('distance_km')
      .to_string(index=False))

=== QUERY 2: MARKETS WITHIN 100KM OF BLANTYRE ===
Total markets     : 113
Within 100km      : 44

           market      district  distance_km  total_spikes  spike_rate_pct
          Chikuli      Blantyre          7.0             9            6.25
            Lunzu      Blantyre          8.3            45           11.69
         Lirangwe      Blantyre         17.4            31            8.45
            Limbe Blantyre City         20.0             0            0.00
  Chiradzulu Boma    Chiradzulu         25.5            11            4.31
          Bvumbwe Blantyre City         32.2            40           11.53
          Thondwe         Zomba         37.4            33           10.51
        Neno Boma          Neno         43.6            29            9.73
          Dyelatu      Chikwawa         43.8            32            9.64
      Mwanza Boma        Mwanza         46.3            29            9.51
           Mayaka         Zomba         46.6            33            9.97
  

In [3]:
# ── SPATIAL QUERY 3 — Monitoring gaps ────────────────────────────────────────
coverage = mkt_count.merge(
    districts[['NAME_1','risk_score','critical_count']],
    on='NAME_1',
    how='right'
).fillna(0)

coverage['markets_per_risk'] = (
    coverage['market_count'] / (coverage['risk_score_y'] + 1)
).round(4)

coverage = coverage.sort_values('markets_per_risk')

print('=== QUERY 3: MONITORING GAPS ===')
print('Low ratio = high risk, few markets = hidden crisis zone')
print()
print(coverage[['NAME_1','risk_score_y','market_count','markets_per_risk']]
      .head(10)
      .rename(columns={
          'NAME_1'          : 'district',
          'risk_score_y'    : 'risk_score',
          'market_count'    : 'markets',
          'markets_per_risk': 'coverage_ratio'
      })
      .to_string(index=False))

=== QUERY 3: MONITORING GAPS ===
Low ratio = high risk, few markets = hidden crisis zone

district  risk_score  markets  coverage_ratio
  Likoma           0      0.0          0.0000
 Mulanje         277      5.0          0.0180
Machinga         436      8.0          0.0183
    Neno          53      1.0          0.0185
   Zomba         262      5.0          0.0190
 Karonga         104      2.0          0.0190
  Mwanza          48      1.0          0.0204
  Rumphi          92      2.0          0.0215
Blantyre         181      4.0          0.0220
Chikwawa         217      5.0          0.0229


In [4]:
# Save all three query results
mkt_count.to_csv('query1_markets_per_district.csv', index=False)
nearby[['market','district','distance_km','total_spikes','spike_rate_pct']]\
      .sort_values('distance_km')\
      .to_csv('query2_markets_100km_blantyre.csv', index=False)
coverage[['NAME_1','risk_score_y','market_count','markets_per_risk']]\
        .rename(columns={'NAME_1':'district','risk_score_y':'risk_score',
                         'market_count':'markets','markets_per_risk':'coverage_ratio'})\
        .to_csv('query3_monitoring_gaps.csv', index=False)

print('All three query results saved')

All three query results saved
